#Phase 2: Data Understanding - NOSDRA Oil Spill Risk Classifier

In [ ]:
import pandas as pd
import numpy as np
file_path = "/content/drive/MyDrive/nosdra_2026-08-23_15_32_02UTC_complete.csv"
df = pd.read_csv(file_path, on_bad_lines='skip') # Using on_bad_lines='skip' allows the file to load by bypassing problematic rows.

# Preview the first 5 rows to verify successful ingestion
df.head()

,id,updatefor,status,zonaloffice,company,incidentnumber,incidentdate,reportdate,contaminant,estimatedquantity,...,postcleanupinspectiondate,postimpactassessmentdate,remediationstart,remediationend,remediationtype,finalsamplingdate,finallabresultsdate,certificatedate,certificatenumber,lastupdatedby
0,2,NaN,confirmed,NaN,ADDAX,HSE/OBO/0611/101,2006-11-23,NaN,cr,225,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOSDRA
1,3,NaN,confirmed,NaN,ADDAX,HSE/OBO/0612/108,2006-12-18,NaN,cr,0.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOSDRA
2,4,NaN,confirmed,NaN,ADDAX,HSE/OBO/0612/110,2006-12-27,NaN,cr,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOSDRA
3,5,NaN,confirmed,NaN,ADDAX,HSE/OBO/0706/166,2007-05-14,NaN,cr,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOSDRA
4,6,NaN,confirmed,NaN,ADDAX,HSE/OBO/0708/201,2007-08-16,NaN,gs,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOSDRA


## 1. Data Ingestion

In [ ]:
file_path = "/content/drive/MyDrive/nosdra_2026-08-23_15_32_02UTC_complete.csv"
df = pd.read_csv(file_path, on_bad_lines='skip')

print("--- INITIAL STRUCTURAL INSPECTION ---")
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")


--- INITIAL STRUCTURAL INSPECTION ---
Dataset Shape: 21107 rows, 42 columns



##2. Missing Value Assessment

In [ ]:
print("--- MISSING VALUE ASSESSMENT ---")
missing_data = df.isnull().sum()
missing_pct = (missing_data / len(df)) * 100

# Identify columns with > 20% missing values
cols_over_20 = missing_pct[missing_pct > 20]
print("Columns exceeding 20% missing data threshold:")
if not cols_over_20.empty:
    for col, pct in cols_over_20.items():
        print(f" - {col}: {pct:.2f}%")
else:
    print(" - None")
print("\n")

--- MISSING VALUE ASSESSMENT ---
Columns exceeding 20% missing data threshold:
 - updatefor: 26.34%
 - zonaloffice: 30.99%
 - incidentnumber: 22.85%
 - reportdate: 43.49%
 - estimatedquantity: 36.95%
 - quantityrecovered: 86.61%
 - spillstopdate: 71.40%
 - typeoffacility: 37.91%
 - initialcontainmentmeasures: 74.29%
 - latitude: 22.15%
 - longitude: 22.15%
 - lga: 25.79%
 - estimatedspillarea: 75.75%
 - descriptionofimpact: 45.67%
 - attachments: 20.92%
 - formadate: 65.86%
 - formbdate: 72.54%
 - formcdate: 89.56%
 - jivdate: 22.34%
 - jivpresent: 44.94%
 - cleanupdate: 91.20%
 - cleanupcompleteddate: 91.60%
 - cleanupmethods: 90.76%
 - postcleanupinspectiondate: 85.09%
 - postimpactassessmentdate: 99.65%
 - remediationstart: 97.58%
 - remediationend: 97.82%
 - remediationtype: 97.42%
 - finalsamplingdate: 91.05%
 - finallabresultsdate: 98.46%
 - certificatedate: 90.13%
 - certificatenumber: 90.29%




## 3. Data Dictionary Generation

In [ ]:
print("--- KNOWN DATA LIMITATIONS (BUSINESS LOGIC) ---")
print("1. Temporal Sparsity: Published data does not reliably include oil spills prior to 2006.")
print("2. Review Status: Spills currently under active review by NOSDRA are excluded from this download.")
print("3. Recent Spills: The most recent oil spills are not included due to data latency.\n")


--- KNOWN DATA LIMITATIONS (BUSINESS LOGIC) ---
1. Temporal Sparsity: Published data does not reliably include oil spills prior to 2006.
2. Review Status: Spills currently under active review by NOSDRA are excluded from this download.
3. Recent Spills: The most recent oil spills are not included due to data latency.



## 4. Target Variable Integrity

In [ ]:
print("--- TARGET VARIABLE INTEGRITY ---")
# Assuming the target column is named 'severity' or 'category' based on project context
target_col = 'severity' if 'severity' in df.columns.str.lower() else 'category'

if target_col in df.columns.str.lower():
    actual_col_name = [c for c in df.columns if c.lower() == target_col][0]
    unique_classes = df[actual_col_name].unique()
    print(f"Unique values found in '{actual_col_name}': {unique_classes}")
    print("Action Required: Ensure classes map strictly to 'Major', 'Medium', and 'Minor'. Correct any typos (e.g., 'minor', ' minr').\n")
else:
    print("Warning: Target variable column ('severity' or 'category') not found. Please verify column names.\n")


--- TARGET VARIABLE INTEGRITY ---



## 5. Data Dictionary Generation


In [ ]:
from IPython.display import display

# Create a dictionary to map specific intended uses (default to 'Exclude')
intended_uses = {
    'incidentdate': 'Predictive Feature (Engineer Year/Month)',
    'company': 'Predictive Feature',
    'cause': 'Predictive Feature',
    'statesaffected': 'Predictive Feature',
    'lga': 'Predictive Feature',
    'typeoffacility': 'Predictive Feature',
    'jivpresent': 'Predictive Feature',
    'estimatedquantity': 'Target Derivation'
}

data_dictionary = pd.DataFrame({
    'Column Name': df.columns,
    'Data Type': df.dtypes.values,
    'Missing %': missing_pct.values.round(2)
})

# Apply definitions based on column names (you can expand this logic)
data_dictionary['Definition'] = data_dictionary['Column Name'].apply(
    lambda x: "Core Feature" if x in intended_uses else "Administrative/Excluded"
)

# Apply Intended Use mapping, default to 'Exclude (>20% null or non-predictive)'
data_dictionary['Intended Use'] = data_dictionary['Column Name'].map(intended_uses).fillna("Exclude")

display(data_dictionary)

,Column Name,Data Type,Missing %,Definition,Intended Use
0,id,int64,0.00,Administrative/Excluded,Exclude
1,updatefor,float64,26.34,Administrative/Excluded,Exclude
2,status,object,0.00,Administrative/Excluded,Exclude
3,zonaloffice,object,30.99,Administrative/Excluded,Exclude
4,company,object,0.19,Core Feature,Predictive Feature
5,incidentnumber,object,22.85,Administrative/Excluded,Exclude
6,incidentdate,object,3.20,Core Feature,Predictive Feature (Engineer Year/Month)
7,reportdate,object,43.49,Administrative/Excluded,Exclude
8,contaminant,object,11.31,Administrative/Excluded,Exclude
9,estimatedquantity,object,36.95,Core Feature,Target Derivation


### 6. Column Exclusion Rationale & Business Logic

In accordance with the project's Data Preparation methodology, specific features and records have been evaluated for exclusion prior to the modeling phase[cite: 2]. The rationale for these exclusions is documented below:

**Feature Exclusions (Columns)**
*   **Non-Predictive Identifiers:** Columns such as `id` and `incidentnumber` are excluded because they act as unique identifiers and hold no predictive value for the multi-class severity classifier[cite: 2].
*   **Free-Text Descriptions:** Columns containing qualitative, unstructured text, such as `descriptionofimpact` and `attachments`, are removed, as Natural Language Processing (NLP) is outside the scope of this tabular data project[cite: 2].
*   **High Missingness (>20%):** 32 columns are excluded due to having more than 20% missing data, as they lack sufficient data density for reliable median or mode imputation[cite: 2]. These include: `updatefor`, `zonaloffice`, `reportdate`, `estimatedquantity`, `quantityrecovered`, `spillstopdate`, `typeoffacility`, `initialcontainmentmeasures`, `latitude`, `longitude`, `lga`, `estimatedspillarea`, `formadate`, `formbdate`, `formcdate`, `jivdate`, `jivpresent`, `cleanupdate`, `cleanupcompleteddate`, `cleanupmethods`, `postcleanupinspectiondate`, `postimpactassessmentdate`, `remediationstart`, `remediationend`, `remediationtype`, `finalsamplingdate`, `finallabresultsdate`, `certificatedate`, and `certificatenumber`.

**Record Exclusions (Rows)**
*   **Temporal Sparsity:** Records prior to 2006 are excluded due to known data sparsity in the published NOSDRA dataset[cite: 3].
*   **Incomplete Reviews:** Incident records currently under active review by NOSDRA are excluded, as their details are not finalized[cite: 3].
*   **Data Latency:** The most recent spill records are omitted due to inherent reporting latency[cite: 3].